<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIRy–pyOMOP Transformation Fidelity

Encounter restoration and audit correction.

This notebook reuses the completed V0–V5 transformation outputs. It restores the omitted Encounter domain from the original controlled-variant archives, applies the verified V2 and V3 warning rules to the TFL sidecar, and recomputes the same four fidelity metrics.

# Phase A

## Environment

In [1]:
import sys
import subprocess
import importlib
import json
import ast
import uuid
import hashlib
import shutil
import sqlite3
import tempfile
import asyncio
import tarfile
from pathlib import Path
from collections import Counter
from enum import Enum

import numpy as np
import pandas as pd

def ensure_package(import_name, pip_spec):
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            pip_spec,
        ])
        importlib.invalidate_caches()
        return importlib.import_module(import_name)

pyomop = ensure_package("pyomop", "pyomop==6.4.0")
fhiry = ensure_package("fhiry", "fhiry==5.2.2")
ensure_package("nest_asyncio", "nest-asyncio")

import fhiry.parallel as fp

print("Python:", sys.version.split()[0])
print("pyomop:", getattr(pyomop, "__version__", "unknown"))
print("fhiry:", getattr(fhiry, "__version__", "unknown"))
print("pandas:", pd.__version__)

Python: 3.12.13
pyomop: 6.4.0
fhiry: 5.2.2
pandas: 2.2.2


In [2]:
from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

RUN_ROOT = MYDRIVE / "fhir_omop_colab" / "tfl_execution_v6"
NORMALIZED_DIR = RUN_ROOT / "normalized_sources"
ORIGINAL_AUDIT_DIR = RUN_ROOT / "fidelity_audit"
ORIGINAL_DB_DIR = RUN_ROOT / "fresh_omop"

REPAIR_ROOT = RUN_ROOT / "encounter_audit_repair_v10"
CORRECTED_DB_DIR = REPAIR_ROOT / "fresh_omop_corrected"
CORRECTED_AUDIT_DIR = REPAIR_ROOT / "fidelity_audit_corrected"
OUTPUT_DIR = REPAIR_ROOT / "outputs"
ENCOUNTER_FLAT_DIR = REPAIR_ROOT / "encounter_flattened"

RAW_ROOT = Path("/content/drive/MyDrive/MyDrive fhir_omop_colab ")

for p in [
    REPAIR_ROOT,
    CORRECTED_DB_DIR,
    CORRECTED_AUDIT_DIR,
    OUTPUT_DIR,
    ENCOUNTER_FLAT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo")

if not REPO_DIR.exists():
    rc = subprocess.call([
        "git",
        "clone",
        "-q",
        REPO_URL,
        str(REPO_DIR),
    ])
    if rc != 0:
        print("Repository clone unavailable; Drive outputs will still be written.")

print("RUN_ROOT:", RUN_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("REPAIR_ROOT:", REPAIR_ROOT)

Mounted at /content/drive
RUN_ROOT: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6
RAW_ROOT: /content/drive/MyDrive/MyDrive fhir_omop_colab 
REPAIR_ROOT: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10


## Existing outputs

In [3]:
VARIANTS = ["V0", "V1", "V2", "V3", "V4", "V5"]

def pick_normalized_file(variant):
    names = [
        f"{variant}_normalized.parquet",
        f"{variant}_recovered_normalized.parquet",
        f"{variant}_constructed_normalized.parquet",
    ]
    if variant == "V0":
        names.insert(0, "V0_recovered_normalized.parquet")
    if variant == "V5":
        names.insert(0, "V5_constructed_normalized.parquet")

    for name in names:
        p = NORMALIZED_DIR / name
        if p.exists() and p.stat().st_size > 0:
            return p
    return None

SOURCE_DF = {}
TFL_AUDIT = {}
ORIGINAL_DB = {}

for variant in VARIANTS:
    source_path = pick_normalized_file(variant)
    if source_path is None:
        raise FileNotFoundError(
            f"Normalized source not found for {variant}: {NORMALIZED_DIR}"
        )

    audit_path = ORIGINAL_AUDIT_DIR / f"{variant}_fidelity_audit.parquet"
    db_path = ORIGINAL_DB_DIR / f"{variant}_tfl.sqlite"

    if not audit_path.exists():
        raise FileNotFoundError(audit_path)
    if not db_path.exists():
        raise FileNotFoundError(db_path)

    SOURCE_DF[variant] = pd.read_parquet(source_path)
    TFL_AUDIT[variant] = pd.read_parquet(audit_path)
    ORIGINAL_DB[variant] = db_path

status = pd.DataFrame([
    {
        "variant": variant,
        "normalized_rows": len(SOURCE_DF[variant]),
        "audit_rows": len(TFL_AUDIT[variant]),
        "original_database_mb": ORIGINAL_DB[variant].stat().st_size / (1024 ** 2),
    }
    for variant in VARIANTS
])

display(status)

,variant,normalized_rows,audit_rows,original_database_mb
0,V0,129333,154333,23.429688
1,V1,129333,154333,23.433594
2,V2,129333,154333,23.429688
3,V3,129333,154333,23.484375
4,V4,126507,151507,23.460938
5,V5,129333,154333,23.488281


# Phase B

## Raw Encounter resources

In [4]:
ARCHIVES = {
    "V1": RAW_ROOT / "V1_missing_demographics_clinical_core_25k.tar.gz",
    "V2": RAW_ROOT / "V2_duplicate_encounter_ids_clinical_core_25k.tar.gz",
    "V3": RAW_ROOT / "V3_conflicting_codings_clinical_core_25k.tar.gz",
    "V4": RAW_ROOT / "V4_missing_medications_clinical_core_25k.tar.gz",
}

missing_archives = [
    variant
    for variant, path in ARCHIVES.items()
    if not path.exists()
]

if missing_archives:
    raise FileNotFoundError(
        "Missing source archive(s): " + ", ".join(missing_archives)
    )

def read_encounter_resources(archive_path):
    resources = []

    with tarfile.open(archive_path, mode="r:gz") as tar:
        members = [
            member
            for member in tar.getmembers()
            if member.isfile()
            and member.name.lower().endswith(".ndjson")
        ]

        for member in members:
            handle = tar.extractfile(member)
            if handle is None:
                continue

            for raw_line in handle:
                try:
                    obj = json.loads(
                        raw_line.decode("utf-8").strip()
                    )
                except Exception:
                    continue

                if obj.get("resourceType") == "Encounter":
                    resources.append(obj)

                elif obj.get("resourceType") == "Bundle":
                    for entry in obj.get("entry", []):
                        resource = entry.get("resource", {})
                        if resource.get("resourceType") == "Encounter":
                            resources.append(resource)

    return resources

RAW_ENCOUNTERS = {
    variant: read_encounter_resources(path)
    for variant, path in ARCHIVES.items()
}

def duplicate_summary(resources):
    ids = [
        str(resource.get("id"))
        for resource in resources
        if resource.get("id") is not None
    ]

    counts = Counter(ids)
    duplicate_counts = {
        encounter_id: count
        for encounter_id, count in counts.items()
        if count > 1
    }

    return {
        "encounter_rows": len(resources),
        "rows_with_id": len(ids),
        "unique_encounter_ids": len(counts),
        "duplicate_id_values": len(duplicate_counts),
        "duplicate_rows": sum(duplicate_counts.values()),
    }

raw_summary = pd.DataFrame([
    {
        "variant": variant,
        **duplicate_summary(RAW_ENCOUNTERS[variant]),
    }
    for variant in ["V1", "V2", "V3", "V4"]
])

display(raw_summary)

for variant in ["V1", "V2", "V3", "V4"]:
    if len(RAW_ENCOUNTERS[variant]) != 25000:
        raise RuntimeError(
            f"{variant}: expected 25,000 Encounter resources, "
            f"found {len(RAW_ENCOUNTERS[variant]):,}."
        )

controls = raw_summary[
    raw_summary["variant"].isin(["V1", "V3", "V4"])
]

if not (controls["duplicate_rows"] == 0).all():
    raise RuntimeError(
        "An unaffected Encounter control contains duplicate Encounter IDs."
    )

v2_dup_rows = int(
    raw_summary.loc[
        raw_summary["variant"] == "V2",
        "duplicate_rows",
    ].iloc[0]
)

if v2_dup_rows != 2393:
    raise RuntimeError(
        f"V2 duplicate-row evidence changed: expected 2,393, found {v2_dup_rows:,}."
    )

print("PASS: V2 raw Encounter duplication = 2,393 rows.")

,variant,encounter_rows,rows_with_id,unique_encounter_ids,duplicate_id_values,duplicate_rows
0,V1,25000,25000,25000,0,0
1,V2,25000,25000,23790,1183,2393
2,V3,25000,25000,25000,0,0
3,V4,25000,25000,25000,0,0


PASS: V2 raw Encounter duplication = 2,393 rows.


## V0 and V5 Encounter baseline

In [5]:
def canonical_resource(resource):
    return json.dumps(
        resource,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

def canonical_control_map(resources):
    output = {}

    for resource in resources:
        rid = resource.get("id")

        if rid is None:
            raise RuntimeError(
                "An unaffected Encounter resource has no id."
            )

        rid = str(rid)

        if rid in output:
            raise RuntimeError(
                f"Duplicate Encounter id in unaffected control: {rid}"
            )

        output[rid] = canonical_resource(resource)

    return output

control_maps = {
    variant: canonical_control_map(RAW_ENCOUNTERS[variant])
    for variant in ["V1", "V3", "V4"]
}

if not (
    control_maps["V1"] == control_maps["V3"]
    and control_maps["V1"] == control_maps["V4"]
):
    raise RuntimeError(
        "V1, V3, and V4 Encounter resources are not identical. "
        "V0 Encounter recovery stopped."
    )

RAW_ENCOUNTERS["V0"] = [
    json.loads(canonical_resource(resource))
    for resource in RAW_ENCOUNTERS["V1"]
]

# V5 combines V1 Patient changes with V3 Condition changes only.
RAW_ENCOUNTERS["V5"] = [
    json.loads(canonical_resource(resource))
    for resource in RAW_ENCOUNTERS["V1"]
]

print("PASS: V1, V3, and V4 Encounter controls are identical.")
print("V0 Encounter source: verified unaffected control consensus")
print("V5 Encounter source: V0 Encounter baseline")

PASS: V1, V3, and V4 Encounter controls are identical.
V0 Encounter source: verified unaffected control consensus
V5 Encounter source: V0 Encounter baseline


# Phase C

## FHIRy Encounter normalization

In [6]:
FHIRY_CONFIG = {
    "REMOVE": ["text.div", "meta"],
    "RENAME": {},
}

ENCOUNTER_DF = {}

for variant in VARIANTS:
    variant_dir = ENCOUNTER_FLAT_DIR / variant

    if variant_dir.exists():
        shutil.rmtree(variant_dir)

    variant_dir.mkdir(parents=True, exist_ok=True)

    ndjson_path = variant_dir / "Encounter.ndjson"

    with open(ndjson_path, "w", encoding="utf-8") as handle:
        for resource in RAW_ENCOUNTERS[variant]:
            handle.write(
                json.dumps(resource, ensure_ascii=False)
                + "\n"
            )

    df = fp.ndjson(
        str(variant_dir),
        config_json=json.dumps(FHIRY_CONFIG),
    )

    if df is None or len(df) != 25000:
        raise RuntimeError(
            f"{variant}: FHIRy produced "
            f"{0 if df is None else len(df):,} Encounter rows."
        )

    if "resourceType" not in df.columns:
        raise RuntimeError(
            f"{variant}: FHIRy output has no resourceType column."
        )

    if "id" not in df.columns:
        raise RuntimeError(
            f"{variant}: FHIRy output has no Encounter id column."
        )

    if "patientId" not in df.columns:
        reference_col = None

        for candidate in [
            "subject.reference",
            "resource.subject.reference",
        ]:
            if candidate in df.columns:
                reference_col = candidate
                break

        if reference_col is None:
            raise RuntimeError(
                f"{variant}: patientId and subject.reference are both unavailable."
            )

        df["patientId"] = (
            df[reference_col]
            .astype("string")
            .str.rstrip("/")
            .str.split("/")
            .str[-1]
        )

    raw_ids = [
        str(resource.get("id"))
        for resource in RAW_ENCOUNTERS[variant]
    ]

    flat_ids = df["id"].astype(str).tolist()

    if flat_ids != raw_ids:
        raise RuntimeError(
            f"{variant}: FHIRy changed Encounter row order; "
            "target-ID order linking is not safe."
        )

    df["__tfl_resource_type"] = "Encounter"
    df["__tfl_resource_id"] = df["id"].astype("string")
    df["__tfl_encounter_sequence"] = np.arange(
        len(df),
        dtype=np.int64,
    )

    ENCOUNTER_DF[variant] = df

    df.to_parquet(
        REPAIR_ROOT / f"{variant}_Encounter_normalized.parquet",
        index=False,
    )

normalization_summary = pd.DataFrame([
    {
        "variant": variant,
        "rows": len(ENCOUNTER_DF[variant]),
        "patient_id_nonmissing": int(
            ENCOUNTER_DF[variant]["patientId"].notna().sum()
        ),
        "unique_encounter_ids": int(
            ENCOUNTER_DF[variant]["id"].astype(str).nunique()
        ),
        "duplicate_rows": int(
            ENCOUNTER_DF[variant]["id"]
            .astype("string")
            .duplicated(keep=False)
            .sum()
        ),
    }
    for variant in VARIANTS
])

display(normalization_summary)

Processing NDJSON files: 100%|██████████| 1/1 [05:36<00:00, 336.62s/it]


,variant,rows,patient_id_nonmissing,unique_encounter_ids,duplicate_rows
0,V0,25000,25000,25000,0
1,V1,25000,25000,25000,0
2,V2,25000,25000,23790,2393
3,V3,25000,25000,25000,0
4,V4,25000,25000,25000,0
5,V5,25000,25000,25000,0


# Phase D

## Visit mapping

In [7]:
PYOMOP_DIR = Path(pyomop.__file__).resolve().parent
MAPPING_PATH = PYOMOP_DIR / "mapping.default.json"

if not MAPPING_PATH.exists():
    candidates = list(PYOMOP_DIR.rglob("mapping.default.json"))

    if not candidates:
        raise FileNotFoundError(
            "pyOMOP mapping.default.json was not found."
        )

    MAPPING_PATH = sorted(
        candidates,
        key=lambda p: len(
            p.relative_to(PYOMOP_DIR).parts
        ),
    )[0]

with open(
    MAPPING_PATH,
    "r",
    encoding="utf-8",
) as handle:
    FULL_MAPPING = json.load(handle)

VISIT_TABLE_RULES = [
    rule
    for rule in FULL_MAPPING.get("tables", [])
    if rule.get("name") == "visit_occurrence"
]

if len(VISIT_TABLE_RULES) != 1:
    raise RuntimeError(
        f"Expected one visit_occurrence mapping rule, "
        f"found {len(VISIT_TABLE_RULES)}."
    )

VISIT_RULE = VISIT_TABLE_RULES[0]

visit_rule_order = next(
    index
    for index, rule in enumerate(
        FULL_MAPPING.get("tables", []),
        start=1,
    )
    if rule is VISIT_RULE
)

VISIT_MAPPING_RULE_ID = (
    f"Encounter_to_visit_occurrence_r{visit_rule_order:02d}"
)

VISIT_MAPPING = {
    "csv_key": FULL_MAPPING.get(
        "csv_key",
        "patientId",
    ),
    "tables": [VISIT_RULE],
    "concept": [
        item
        for item in FULL_MAPPING.get("concept", [])
        if item.get("table") == "visit_occurrence"
    ],
    "force_text_fields": FULL_MAPPING.get(
        "force_text_fields",
        [],
    ),
}

VISIT_MAPPING_PATH = (
    REPAIR_ROOT
    / "mapping.visit_occurrence.v10.json"
)

with open(
    VISIT_MAPPING_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        VISIT_MAPPING,
        handle,
        indent=2,
    )

print("Mapping rule:", VISIT_MAPPING_RULE_ID)
print("Mapping:", VISIT_MAPPING_PATH)

display(
    pd.DataFrame([
        {
            "target_field": target,
            "source": source,
        }
        for target, source
        in VISIT_RULE["columns"].items()
    ])
)

Mapping rule: Encounter_to_visit_occurrence_r02
Mapping: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/mapping.visit_occurrence.v10.json


,target_field,source
0,person_id,patientId
1,visit_start_date,period.start
2,visit_start_datetime,period.start
3,visit_end_date,period.end
4,visit_end_datetime,period.end
5,visit_concept_id,{'const': 0}
6,visit_type_concept_id,{'const': 0}
7,visit_source_value,class.code
8,visit_source_concept_id,{'const': 0}
9,admitted_from_source_value,


## Corrected databases

In [8]:
from pyomop import CdmEngineFactory
from pyomop.loader import CdmCsvLoader

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()

        if loop.is_running():
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(coro)

    except RuntimeError:
        pass

    return asyncio.run(coro)


async def add_encounters_to_existing_db(
    encounter_df,
    db_path,
):
    cdm = CdmEngineFactory(
        db="sqlite",
        name=str(db_path),
    )

    _ = cdm.engine

    with tempfile.NamedTemporaryFile(
        suffix=".csv",
        delete=False,
    ) as tmp:
        csv_path = Path(tmp.name)

    try:
        encounter_df.to_csv(
            csv_path,
            index=False,
        )

        loader = CdmCsvLoader(
            cdm,
            version="cdm54",
        )

        await loader.load(
            csv_path=str(csv_path),
            mapping_path=str(VISIT_MAPPING_PATH),
            chunk_size=500,
        )

    finally:
        if csv_path.exists():
            csv_path.unlink()

        await cdm.dispose()


CORRECTED_DB = {}
visit_rows = []

for variant in VARIANTS:
    source_db = ORIGINAL_DB[variant]

    corrected_db = (
        CORRECTED_DB_DIR
        / f"{variant}_tfl_encounter_corrected.sqlite"
    )

    shutil.copy2(
        source_db,
        corrected_db,
    )

    with sqlite3.connect(corrected_db) as conn:
        before = conn.execute(
            "SELECT COUNT(*) FROM visit_occurrence"
        ).fetchone()[0]

        person_n = conn.execute(
            "SELECT COUNT(*) FROM person"
        ).fetchone()[0]

    if before != 0:
        raise RuntimeError(
            f"{variant}: copied database already has "
            f"{before:,} visit rows."
        )

    if person_n != 1071:
        raise RuntimeError(
            f"{variant}: expected 1,071 persons, "
            f"found {person_n:,}."
        )

    print(
        "Loading Encounter -> visit_occurrence:",
        variant
    )

    run_async(
        add_encounters_to_existing_db(
            ENCOUNTER_DF[variant],
            corrected_db,
        )
    )

    with sqlite3.connect(corrected_db) as conn:
        after = conn.execute(
            "SELECT COUNT(*) FROM visit_occurrence"
        ).fetchone()[0]

        unique_target_ids = conn.execute(
            "SELECT COUNT(DISTINCT visit_occurrence_id) "
            "FROM visit_occurrence"
        ).fetchone()[0]

        invalid_person_links = conn.execute(
            """
            SELECT COUNT(*)
            FROM visit_occurrence v
            LEFT JOIN person p
              ON p.person_id = v.person_id
            WHERE p.person_id IS NULL
            """
        ).fetchone()[0]

    if after != 25000:
        raise RuntimeError(
            f"{variant}: expected 25,000 visit rows, "
            f"found {after:,}."
        )

    if unique_target_ids != 25000:
        raise RuntimeError(
            f"{variant}: visit_occurrence_id is not unique."
        )

    if invalid_person_links != 0:
        raise RuntimeError(
            f"{variant}: {invalid_person_links:,} visit rows "
            "have invalid person links."
        )

    CORRECTED_DB[variant] = corrected_db

    visit_rows.append({
        "variant": variant,
        "visit_rows_before": before,
        "visit_rows_after": after,
        "unique_visit_occurrence_ids": unique_target_ids,
        "invalid_person_links": invalid_person_links,
        "database_mb": (
            corrected_db.stat().st_size
            / (1024 ** 2)
        ),
    })

visit_database_summary = pd.DataFrame(
    visit_rows
)

display(visit_database_summary)

visit_database_summary.to_csv(
    OUTPUT_DIR
    / "visit_occurrence_database_summary.csv",
    index=False,
)

Loading Encounter -> visit_occurrence: V0
Loading Encounter -> visit_occurrence: V1
Loading Encounter -> visit_occurrence: V2
Loading Encounter -> visit_occurrence: V3
Loading Encounter -> visit_occurrence: V4
Loading Encounter -> visit_occurrence: V5


,variant,visit_rows_before,visit_rows_after,unique_visit_occurrence_ids,invalid_person_links,database_mb
0,V0,0,25000,25000,0,27.531250
1,V1,0,25000,25000,0,27.535156
2,V2,0,25000,25000,0,27.531250
3,V3,0,25000,25000,0,27.585938
4,V4,0,25000,25000,0,27.562500
5,V5,0,25000,25000,0,27.589844


# Phase E

## Encounter audit records

In [9]:
class Fate(str, Enum):
    PRESERVED = "PRESERVED"
    NORMALIZED = "NORMALIZED"
    COLLAPSED = "COLLAPSED"
    AMBIGUOUS = "AMBIGUOUS"
    UNMAPPED = "UNMAPPED"
    DROPPED = "DROPPED"
    TRACEABILITY_LOSS = "TRACEABILITY_LOSS"


def split_warnings(value):
    if value is None:
        return []

    return [
        item
        for item in str(value).split("|")
        if item and item != "nan"
    ]


def warning_string(values):
    return "|".join(
        sorted(set(values))
    )


existing_version_values = (
    pd.concat(
        [
            audit["transformation_version"].dropna()
            for audit in TFL_AUDIT.values()
            if "transformation_version" in audit.columns
        ],
        ignore_index=True,
    )
    .astype(str)
    .unique()
)

BASE_TRANSFORMATION_VERSION = (
    existing_version_values[0]
    if len(existing_version_values)
    else "TFL-1.2"
)

TRANSFORMATION_VERSION = (
    BASE_TRANSFORMATION_VERSION
    + "|encounter-repair-v10"
)

visit_source_paths = "|".join(
    sorted({
        value
        for value in VISIT_RULE["columns"].values()
        if isinstance(value, str) and value
    })
)

visit_target_fields = "|".join(
    sorted(
        VISIT_RULE["columns"].keys()
    )
)


def target_visit_ids(db_path):
    with sqlite3.connect(db_path) as conn:
        return [
            int(row[0])
            for row in conn.execute(
                """
                SELECT visit_occurrence_id
                FROM visit_occurrence
                ORDER BY visit_occurrence_id
                """
            ).fetchall()
        ]


def build_encounter_audit(
    variant,
    encounter_df,
    db_path,
    source_index_start,
):
    target_ids = target_visit_ids(
        db_path
    )

    if len(target_ids) != len(
        encounter_df
    ):
        raise RuntimeError(
            f"{variant}: source/target Encounter "
            "row reconciliation failed."
        )

    ids = encounter_df[
        "id"
    ].astype("string")

    duplicate_mask = (
        ids.duplicated(
            keep=False
        )
    )

    rows = []

    for sequence, (
        (_, source_row),
        target_id,
        is_duplicate,
    ) in enumerate(
        zip(
            encounter_df.iterrows(),
            target_ids,
            duplicate_mask.tolist(),
        )
    ):
        source_id = str(
            source_row["id"]
        )

        warnings = []

        status = (
            Fate.NORMALIZED.value
        )

        if is_duplicate:
            warnings.extend([
                "W_DUPLICATE_SOURCE_ID",
                "W_TRACEABILITY_LOSS",
            ])

            status = (
                Fate.TRACEABILITY_LOSS.value
            )

        transformation_name = (
            f"{variant}|Encounter|{source_id}|"
            f"{sequence}|{target_id}|"
            f"{VISIT_MAPPING_RULE_ID}|"
            f"{TRANSFORMATION_VERSION}"
        )

        rows.append({
            "transformation_id": str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    transformation_name,
                )
            ),
            "variant": variant,
            "source_resource_type": "Encounter",
            "source_resource_id": source_id,
            "source_reference_or_path": visit_source_paths,
            "source_value_or_code": None,
            "target_omop_table": "visit_occurrence",
            "target_omop_record_id": target_id,
            "target_omop_field": visit_target_fields,
            "mapping_rule_id": VISIT_MAPPING_RULE_ID,
            "fidelity_status": status,
            "warning_code": warning_string(
                warnings
            ),
            "transformation_version": TRANSFORMATION_VERSION,
            "source_df_index": int(
                source_index_start
                + sequence
            ),
            "duplicate_source_id": bool(
                is_duplicate
            ),
            "is_mapping_event": True,
            "target_link_method": (
                "encounter_only_pyomop_"
                "verified_source_order"
            ),
        })

    return pd.DataFrame(
        rows
    )


ENCOUNTER_AUDIT = {}

for variant in VARIANTS:
    old_audit = TFL_AUDIT[
        variant
    ]

    numeric_index = (
        pd.to_numeric(
            old_audit[
                "source_df_index"
            ],
            errors="coerce",
        )
        if "source_df_index"
        in old_audit.columns
        else pd.Series(
            dtype=float
        )
    )

    start = (
        int(
            numeric_index.max()
        ) + 1
        if len(
            numeric_index
        )
        and numeric_index.notna().any()
        else len(
            SOURCE_DF[
                variant
            ]
        )
    )

    ENCOUNTER_AUDIT[
        variant
    ] = build_encounter_audit(
        variant,
        ENCOUNTER_DF[
            variant
        ],
        CORRECTED_DB[
            variant
        ],
        start,
    )


encounter_audit_summary = pd.DataFrame([
    {
        "variant": variant,
        "audit_rows": len(
            ENCOUNTER_AUDIT[
                variant
            ]
        ),
        "duplicate_warning_rows": int(
            ENCOUNTER_AUDIT[
                variant
            ][
                "warning_code"
            ]
            .astype(str)
            .str.contains(
                "W_DUPLICATE_SOURCE_ID",
                regex=False,
            )
            .sum()
        ),
        "traceability_loss_rows": int(
            (
                ENCOUNTER_AUDIT[
                    variant
                ][
                    "fidelity_status"
                ]
                ==
                Fate.TRACEABILITY_LOSS.value
            ).sum()
        ),
    }
    for variant in VARIANTS
])

display(
    encounter_audit_summary
)

v2_audit_duplicates = int(
    encounter_audit_summary.loc[
        encounter_audit_summary[
            "variant"
        ]
        ==
        "V2",
        "duplicate_warning_rows",
    ].iloc[0]
)

if v2_audit_duplicates != 2393:
    raise RuntimeError(
        "V2 Encounter audit did not reproduce "
        "2,393 duplicate-warning rows."
    )

,variant,audit_rows,duplicate_warning_rows,traceability_loss_rows
0,V0,25000,0,0
1,V1,25000,0,0
2,V2,25000,2393,2393
3,V3,25000,0,0
4,V4,25000,0,0
5,V5,25000,0,0


## V3 warning correction

In [10]:
V3_CONDITION_CODING_FIELD = (
    "code.coding.codes"
)

for variant in VARIANTS:
    if (
        V3_CONDITION_CODING_FIELD
        not in SOURCE_DF[
            variant
        ].columns
    ):
        raise RuntimeError(
            f"{variant}: "
            f"{V3_CONDITION_CODING_FIELD} "
            "is unavailable."
        )


def condition_subset(df):
    return df[
        df[
            "__tfl_resource_type"
        ].astype(str)
        ==
        "Condition"
    ].copy()


def flatten_tokens(value):
    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):
        output = []

        for item in value:
            output.extend(
                flatten_tokens(
                    item
                )
            )

        return output

    text = str(
        value
    ).strip()

    if (
        text == ""
        or text.lower()
        in {
            "nan",
            "none",
            "null",
            "<na>",
        }
    ):
        return []

    if (
        (
            text.startswith(
                "["
            )
            and text.endswith(
                "]"
            )
        )
        or (
            text.startswith(
                "("
            )
            and text.endswith(
                ")"
            )
        )
    ):
        try:
            parsed = (
                ast.literal_eval(
                    text
                )
            )

            if isinstance(
                parsed,
                (
                    list,
                    tuple,
                    set,
                ),
            ):
                return flatten_tokens(
                    parsed
                )

        except Exception:
            pass

    return [
        text
    ]


def token_set(series):
    values = set()

    for value in series:
        values.update(
            flatten_tokens(
                value
            )
        )

    return {
        str(value).strip()
        for value in values
        if str(value).strip()
    }


condition_token_sets = {
    variant: token_set(
        condition_subset(
            SOURCE_DF[
                variant
            ]
        )[
            V3_CONDITION_CODING_FIELD
        ]
    )
    for variant in VARIANTS
}


coding_cardinality = pd.DataFrame([
    {
        "variant": variant,
        "unique_coding_representations": len(
            condition_token_sets[
                variant
            ]
        ),
    }
    for variant in VARIANTS
])

display(
    coding_cardinality
)


expected_cardinality = {
    "V0": 247,
    "V1": 247,
    "V2": 247,
    "V3": 412,
    "V4": 247,
    "V5": 412,
}

for variant, expected in (
    expected_cardinality.items()
):
    actual = len(
        condition_token_sets[
            variant
        ]
    )

    if actual != expected:
        raise RuntimeError(
            f"{variant}: expected "
            f"{expected} Condition coding "
            f"representations, found {actual}."
        )


V0_CONDITION_CODES = (
    condition_token_sets[
        "V0"
    ]
)

CONDITION_CONFLICT_MASK = {}

for variant in VARIANTS:
    condition_df = (
        condition_subset(
            SOURCE_DF[
                variant
            ]
        )
    )

    flags = condition_df[
        V3_CONDITION_CODING_FIELD
    ].apply(
        lambda value: bool(
            set(
                flatten_tokens(
                    value
                )
            )
            -
            V0_CONDITION_CODES
        )
    )

    CONDITION_CONFLICT_MASK[
        variant
    ] = flags


condition_warning_summary = pd.DataFrame([
    {
        "variant": variant,
        "condition_rows": len(
            CONDITION_CONFLICT_MASK[
                variant
            ]
        ),
        "conflicting_coding_rows": int(
            CONDITION_CONFLICT_MASK[
                variant
            ].sum()
        ),
        "warning_rate": float(
            CONDITION_CONFLICT_MASK[
                variant
            ].mean()
        ),
    }
    for variant in VARIANTS
])

display(
    condition_warning_summary
)


expected_condition_warning_rows = {
    "V0": 0,
    "V1": 0,
    "V2": 0,
    "V3": 2510,
    "V4": 0,
    "V5": 2510,
}

for variant, expected in (
    expected_condition_warning_rows.items()
):
    actual = int(
        condition_warning_summary.loc[
            condition_warning_summary[
                "variant"
            ]
            ==
            variant,
            "conflicting_coding_rows",
        ].iloc[0]
    )

    if actual != expected:
        raise RuntimeError(
            f"{variant}: expected "
            f"{expected:,} V3 warning rows, "
            f"found {actual:,}."
        )

print(
    "PASS: V3/V5 coding warning "
    "pattern reproduced."
)

,variant,unique_coding_representations
0,V0,247
1,V1,247
2,V2,247
3,V3,412
4,V4,247
5,V5,412


,variant,condition_rows,conflicting_coding_rows,warning_rate
0,V0,25000,0,0.0000
1,V1,25000,0,0.0000
2,V2,25000,0,0.0000
3,V3,25000,2510,0.1004
4,V4,25000,0,0.0000
5,V5,25000,2510,0.1004


PASS: V3/V5 coding warning pattern reproduced.


# Phase F

## Corrected TFL audit

In [11]:
mapping_rule_rows = []

for ordinal, rule in enumerate(
    FULL_MAPPING.get("tables", []),
    start=1,
):
    resource_type = None

    for flt in (rule.get("filters", []) or []):
        if (
            flt.get("column") in {
                "resourceType",
                "resource.resourceType",
            }
            and "equals" in flt
        ):
            resource_type = str(flt["equals"])
            break

    if resource_type is None:
        continue

    rule_id = (
        f"{resource_type}_to_"
        f"{rule['name']}_r{ordinal:02d}"
    )

    mapping_rule_rows.append({
        "mapping_rule_id": rule_id,
        "target_fields": "|".join(
            sorted(
                rule.get("columns", {}).keys()
            )
        ),
    })

mapping_rules_df = pd.DataFrame(
    mapping_rule_rows
)

def default_mapping_fate(mapping_rule_id):
    row = mapping_rules_df[
        mapping_rules_df["mapping_rule_id"]
        == mapping_rule_id
    ]

    if len(row) == 1:
        target_fields = str(
            row.iloc[0]["target_fields"]
        )

        if "_source_value" in target_fields:
            return Fate.PRESERVED.value

    return Fate.NORMALIZED.value


CORRECTED_AUDIT = {}

for variant in VARIANTS:
    audit = TFL_AUDIT[variant].copy()

    encounter_mask = (
        audit["source_resource_type"].astype(str)
        == "Encounter"
    )

    audit = audit.loc[
        ~encounter_mask
    ].copy()

    condition_mask = (
        audit["source_resource_type"].astype(str)
        == "Condition"
    )

    for ix in audit.index[condition_mask]:
        warnings = [
            warning
            for warning in split_warnings(
                audit.at[ix, "warning_code"]
            )
            if warning != "W_CONFLICTING_CODING"
        ]

        audit.at[ix, "warning_code"] = (
            warning_string(warnings)
        )

        if audit.at[ix, "fidelity_status"] in {
            Fate.AMBIGUOUS.value,
            Fate.COLLAPSED.value,
        }:
            audit.at[ix, "fidelity_status"] = (
                default_mapping_fate(
                    audit.at[ix, "mapping_rule_id"]
                )
            )

    source_condition = condition_subset(
        SOURCE_DF[variant]
    )

    source_indices = set(
        source_condition.loc[
            CONDITION_CONFLICT_MASK[variant],
            "__tfl_source_df_index",
        ].astype(int)
    )

    audit_source_index = pd.to_numeric(
        audit["source_df_index"],
        errors="coerce",
    )

    corrected_condition_mask = (
        (
            audit["source_resource_type"].astype(str)
            == "Condition"
        )
        & audit_source_index.isin(source_indices)
    )

    for ix in audit.index[corrected_condition_mask]:
        warnings = split_warnings(
            audit.at[ix, "warning_code"]
        )

        warnings.append(
            "W_CONFLICTING_CODING"
        )

        audit.at[ix, "warning_code"] = (
            warning_string(warnings)
        )

        audit.at[ix, "fidelity_status"] = (
            Fate.AMBIGUOUS.value
        )

    all_columns = sorted(
        set(audit.columns)
        | set(
            ENCOUNTER_AUDIT[variant].columns
        )
    )

    audit = audit.reindex(
        columns=all_columns
    )

    encounter_audit = (
        ENCOUNTER_AUDIT[variant]
        .reindex(
            columns=all_columns
        )
    )

    audit = pd.concat(
        [
            audit,
            encounter_audit,
        ],
        ignore_index=True,
    )

    CORRECTED_AUDIT[variant] = audit

    audit.to_parquet(
        CORRECTED_AUDIT_DIR
        / f"{variant}_fidelity_audit_v10.parquet",
        index=False,
    )


corrected_audit_summary = pd.DataFrame([
    {
        "variant": variant,
        "audit_rows": len(
            CORRECTED_AUDIT[variant]
        ),
        "encounter_audit_rows": int(
            (
                CORRECTED_AUDIT[variant][
                    "source_resource_type"
                ].astype(str)
                == "Encounter"
            ).sum()
        ),
        "v2_duplicate_warning_rows": int(
            CORRECTED_AUDIT[variant][
                "warning_code"
            ]
            .fillna("")
            .astype(str)
            .str.contains(
                "W_DUPLICATE_SOURCE_ID",
                regex=False,
            )
            .sum()
        ),
        "v3_coding_warning_rows": int(
            CORRECTED_AUDIT[variant][
                "warning_code"
            ]
            .fillna("")
            .astype(str)
            .str.contains(
                "W_CONFLICTING_CODING",
                regex=False,
            )
            .sum()
        ),
    }
    for variant in VARIANTS
])

display(
    corrected_audit_summary
)

,variant,audit_rows,encounter_audit_rows,v2_duplicate_warning_rows,v3_coding_warning_rows
0,V0,179333,25000,0,0
1,V1,179333,25000,0,0
2,V2,179333,25000,2393,0
3,V3,179333,25000,0,2510
4,V4,176507,25000,0,0
5,V5,179333,25000,0,2510


# Phase G

## Primary metrics

In [12]:
LOSS_STATUSES = {
    Fate.UNMAPPED.value,
    Fate.DROPPED.value,
    Fate.TRACEABILITY_LOSS.value,
}

def metric_rows_for_audit(audit, variant):
    work = audit.copy()

    mapping_event = (
        work["is_mapping_event"]
        .fillna(False)
        .astype(bool)
    )

    traceability_ok = (
        work["fidelity_status"]
        != Fate.TRACEABILITY_LOSS.value
    )

    source_id_present = (
        work["source_resource_id"]
        .notna()
    )

    target_id_present = (
        work["target_omop_record_id"]
        .notna()
    )

    work["has_warning"] = (
        work["warning_code"]
        .fillna("")
        .astype(str)
        .str.len()
        > 0
    )

    work["lineage_valid"] = (
        mapping_event
        & traceability_ok
        & source_id_present
        & target_id_present
    )

    work["mapped_target"] = (
        mapping_event
        & target_id_present
    )

    work["is_loss"] = (
        work["fidelity_status"]
        .isin(LOSS_STATUSES)
    )

    work["is_ambiguity_or_warning"] = (
        (
            work["fidelity_status"]
            == Fate.AMBIGUOUS.value
        )
        | work["has_warning"]
    )

    mapped = work[mapping_event]

    return {
        "variant": variant,
        "audit_items": len(work),
        "mapped_items": len(mapped),
        "lineage_coverage": (
            mapped["lineage_valid"].mean()
            if len(mapped)
            else np.nan
        ),
        "source_mapping_coverage": (
            work["mapped_target"].sum() / len(work)
            if len(work)
            else np.nan
        ),
        "transformation_loss_rate": (
            work["is_loss"].mean()
            if len(work)
            else np.nan
        ),
        "ambiguity_warning_rate": (
            work["is_ambiguity_or_warning"].mean()
            if len(work)
            else np.nan
        ),
    }

primary_metrics = pd.DataFrame([
    metric_rows_for_audit(
        CORRECTED_AUDIT[variant],
        variant,
    )
    for variant in VARIANTS
])

display(primary_metrics)

primary_metrics.to_csv(
    OUTPUT_DIR / "tfl_primary_metrics_v10.csv",
    index=False,
)

,variant,audit_items,mapped_items,lineage_coverage,source_mapping_coverage,transformation_loss_rate,ambiguity_warning_rate
0,V0,179333,151071,1.00000,0.842405,0.157595,0.211612
1,V1,179333,151071,1.00000,0.842405,0.157595,0.212510
2,V2,179333,151071,0.98416,0.842405,0.170939,0.224956
3,V3,179333,151071,1.00000,0.842405,0.157595,0.225608
4,V4,176507,151071,1.00000,0.855892,0.144108,0.199873
5,V5,179333,151071,1.00000,0.842405,0.157595,0.226506


## Warning validation

In [13]:
warning_rows = []

for variant, audit in (
    CORRECTED_AUDIT.items()
):
    for row in audit.itertuples():
        for warning in split_warnings(
            row.warning_code
        ):
            warning_rows.append({
                "variant": variant,
                "warning_code": warning,
            })

warning_long = pd.DataFrame(
    warning_rows
)

warning_counts = (
    warning_long.groupby(
        [
            "variant",
            "warning_code",
        ]
    )
    .size()
    .rename(
        "warning_count"
    )
    .reset_index()
)

audit_denominators = {
    variant: len(
        CORRECTED_AUDIT[
            variant
        ]
    )
    for variant in VARIANTS
}

warning_counts[
    "audit_items"
] = (
    warning_counts[
        "variant"
    ].map(
        audit_denominators
    )
)

warning_counts[
    "warning_rate"
] = (
    warning_counts[
        "warning_count"
    ]
    /
    warning_counts[
        "audit_items"
    ]
)

display(
    warning_counts
)

warning_counts.to_csv(
    OUTPUT_DIR
    / "tfl_warning_counts_rates_v10.csv",
    index=False,
)


EXPECTED_WARNINGS = {
    "V1": [
        "W_DEMOGRAPHIC_MISSING",
    ],
    "V2": [
        "W_DUPLICATE_SOURCE_ID",
        "W_TRACEABILITY_LOSS",
    ],
    "V3": [
        "W_CONFLICTING_CODING",
    ],
    "V4": [
        "W_MEDICATION_ATTRIBUTION",
    ],
    "V5": [
        "W_DEMOGRAPHIC_MISSING",
        "W_CONFLICTING_CODING",
    ],
}


warning_lookup = {
    (
        row.variant,
        row.warning_code,
    ): (
        int(
            row.warning_count
        ),
        float(
            row.warning_rate
        ),
    )
    for row
    in warning_counts.itertuples()
}


validation_rows = []

for variant, warnings in (
    EXPECTED_WARNINGS.items()
):
    for warning in warnings:
        (
            v0_count,
            v0_rate,
        ) = warning_lookup.get(
            (
                "V0",
                warning,
            ),
            (
                0,
                0.0,
            ),
        )

        (
            variant_count,
            variant_rate,
        ) = warning_lookup.get(
            (
                variant,
                warning,
            ),
            (
                0,
                0.0,
            ),
        )

        validation_rows.append({
            "variant": variant,
            "expected_warning": warning,
            "v0_count": v0_count,
            "variant_count": variant_count,
            "delta_count": (
                variant_count
                -
                v0_count
            ),
            "v0_rate": v0_rate,
            "variant_rate": variant_rate,
            "delta_rate": (
                variant_rate
                -
                v0_rate
            ),
            "increased_vs_v0": (
                variant_rate
                >
                v0_rate
            ),
        })


variant_validation = pd.DataFrame(
    validation_rows
)

display(
    variant_validation
)

variant_gate = (
    variant_validation.groupby(
        "variant"
    )[
        "increased_vs_v0"
    ]
    .all()
    .reset_index(
        name=(
            "all_expected_"
            "warnings_increased"
        )
    )
)

display(
    variant_gate
)

variant_validation.to_csv(
    OUTPUT_DIR
    / "tfl_variant_warning_validation_v10.csv",
    index=False,
)

variant_gate.to_csv(
    OUTPUT_DIR
    / "tfl_variant_gate_v10.csv",
    index=False,
)

if not variant_gate[
    "all_expected_warnings_increased"
].all():
    raise RuntimeError(
        "One or more controlled warning "
        "validations failed."
    )

print(
    "PASS: V1–V5 expected warning "
    "responses all increase versus V0."
)

,variant,warning_code,warning_count,audit_items,warning_rate
0,V0,W_MEDICATION_ATTRIBUTION,9687,179333,0.054017
1,V0,W_UNMAPPED_RESOURCE,28262,179333,0.157595
2,V1,W_DEMOGRAPHIC_MISSING,161,179333,0.000898
3,V1,W_MEDICATION_ATTRIBUTION,9687,179333,0.054017
4,V1,W_UNMAPPED_RESOURCE,28262,179333,0.157595
5,V2,W_DUPLICATE_SOURCE_ID,2393,179333,0.013344
6,V2,W_MEDICATION_ATTRIBUTION,9687,179333,0.054017
7,V2,W_TRACEABILITY_LOSS,2393,179333,0.013344
8,V2,W_UNMAPPED_RESOURCE,28262,179333,0.157595
9,V3,W_CONFLICTING_CODING,2510,179333,0.013996


,variant,expected_warning,v0_count,variant_count,delta_count,v0_rate,variant_rate,delta_rate,increased_vs_v0
0,V1,W_DEMOGRAPHIC_MISSING,0,161,161,0.000000,0.000898,0.000898,True
1,V2,W_DUPLICATE_SOURCE_ID,0,2393,2393,0.000000,0.013344,0.013344,True
2,V2,W_TRACEABILITY_LOSS,0,2393,2393,0.000000,0.013344,0.013344,True
3,V3,W_CONFLICTING_CODING,0,2510,2510,0.000000,0.013996,0.013996,True
4,V4,W_MEDICATION_ATTRIBUTION,9687,9843,156,0.054017,0.055765,0.001749,True
5,V5,W_DEMOGRAPHIC_MISSING,0,161,161,0.000000,0.000898,0.000898,True
6,V5,W_CONFLICTING_CODING,0,2510,2510,0.000000,0.013996,0.013996,True


,variant,all_expected_warnings_increased
0,V1,True
1,V2,True
2,V3,True
3,V4,True
4,V5,True


PASS: V1–V5 expected warning responses all increase versus V0.


# Phase H

## Encounter traceability

In [14]:
traceability_rows = []

for variant in VARIANTS:
    source_ids = (
        ENCOUNTER_DF[
            variant
        ][
            "id"
        ]
        .astype(str)
    )

    source_duplicate_rows = int(
        source_ids.duplicated(
            keep=False
        ).sum()
    )

    with sqlite3.connect(
        CORRECTED_DB[
            variant
        ]
    ) as conn:
        target_rows = (
            conn.execute(
                "SELECT COUNT(*) "
                "FROM visit_occurrence"
            ).fetchone()[0]
        )

        target_unique_ids = (
            conn.execute(
                "SELECT COUNT("
                "DISTINCT visit_occurrence_id) "
                "FROM visit_occurrence"
            ).fetchone()[0]
        )

    traceability_rows.append({
        "variant": variant,
        "source_encounter_rows": len(
            source_ids
        ),
        "unique_source_encounter_ids": int(
            source_ids.nunique()
        ),
        "duplicate_source_rows": (
            source_duplicate_rows
        ),
        "target_visit_rows": (
            target_rows
        ),
        "unique_target_visit_ids": (
            target_unique_ids
        ),
        "source_target_row_reconciliation": (
            len(
                source_ids
            )
            ==
            target_rows
        ),
    })


visit_traceability_summary = pd.DataFrame(
    traceability_rows
)

display(
    visit_traceability_summary
)

visit_traceability_summary.to_csv(
    OUTPUT_DIR
    / "visit_traceability_summary_v10.csv",
    index=False,
)

v2 = visit_traceability_summary[
    visit_traceability_summary[
        "variant"
    ]
    ==
    "V2"
].iloc[0]

if int(
    v2[
        "duplicate_source_rows"
    ]
) != 2393:
    raise RuntimeError(
        "V2 source duplicate-row "
        "count changed."
    )

if int(
    v2[
        "unique_target_visit_ids"
    ]
) != 25000:
    raise RuntimeError(
        "V2 target visit IDs "
        "are not unique."
    )

print(
    "PASS: V2 retains duplicate "
    "source identity while "
    "visit_occurrence IDs remain unique."
)

,variant,source_encounter_rows,unique_source_encounter_ids,duplicate_source_rows,target_visit_rows,unique_target_visit_ids,source_target_row_reconciliation
0,V0,25000,25000,0,25000,25000,True
1,V1,25000,25000,0,25000,25000,True
2,V2,25000,23790,2393,25000,25000,True
3,V3,25000,25000,0,25000,25000,True
4,V4,25000,25000,0,25000,25000,True
5,V5,25000,25000,0,25000,25000,True


PASS: V2 retains duplicate source identity while visit_occurrence IDs remain unique.


# Phase I

## Export

In [15]:
if REPO_DIR.exists():
    repo_output = (
        REPO_DIR
        / "outputs"
        / "transformation_fidelity_v10"
    )

    repo_output.mkdir(
        parents=True,
        exist_ok=True,
    )

    for file_path in (
        OUTPUT_DIR.glob(
            "*"
        )
    ):
        if file_path.is_file():
            shutil.copy2(
                file_path,
                repo_output
                /
                file_path.name,
            )

    repo_audit = (
        REPO_DIR
        / "outputs"
        / "fidelity_audit_v10"
    )

    repo_audit.mkdir(
        parents=True,
        exist_ok=True,
    )

    public_rows = []

    for variant in VARIANTS:
        audit = CORRECTED_AUDIT[
            variant
        ].copy()

        sample = audit.sample(
            min(
                100,
                len(
                    audit
                ),
            ),
            random_state=42,
        )

        sample[
            "source_resource_id_hash"
        ] = sample[
            "source_resource_id"
        ].apply(
            lambda value: (
                hashlib.sha256(
                    str(
                        value
                    ).encode(
                        "utf-8"
                    )
                ).hexdigest()[:16]
                if pd.notna(
                    value
                )
                else None
            )
        )

        sample = sample.drop(
            columns=[
                column
                for column
                in [
                    "source_resource_id",
                    "source_value_or_code",
                ]
                if column
                in sample.columns
            ]
        )

        public_rows.append(
            sample
        )

    public_audit_sample = pd.concat(
        public_rows,
        ignore_index=True,
    )

    public_audit_sample.to_csv(
        repo_audit
        / "tfl_public_audit_sample_v10.csv",
        index=False,
    )

    print(
        "Repository outputs:",
        repo_output,
    )

    print(
        "Repository audit sample:",
        repo_audit,
    )


package_path = shutil.make_archive(
    str(
        REPAIR_ROOT
        / "FHIRy_pyOMOP_TFL_v10_outputs"
    ),
    "zip",
    OUTPUT_DIR,
)

print(
    "Drive outputs:",
    OUTPUT_DIR,
)

print(
    "Corrected databases:",
    CORRECTED_DB_DIR,
)

print(
    "Corrected private audits:",
    CORRECTED_AUDIT_DIR,
)

print(
    "Output package:",
    package_path,
)

Repository outputs: /content/ohdsi-fhir-omop-showcase-demo/outputs/transformation_fidelity_v10
Repository audit sample: /content/ohdsi-fhir-omop-showcase-demo/outputs/fidelity_audit_v10
Drive outputs: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/outputs
Corrected databases: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected
Corrected private audits: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fidelity_audit_corrected
Output package: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/FHIRy_pyOMOP_TFL_v10_outputs.zip
